In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

# 1. Khởi tạo Spark Session (Không dùng HiveSupport)
from lakehouse_common import create_spark
spark = create_spark("Delivery - 01_ingest_bronze")

# 2. Đọc file CSV từ MinIO
path = "s3a://lakehouse/delivery_yt.csv"
df = spark.read.csv(path, header=True, inferSchema=False, mode="FAILFAST")

from lakehouse_common import require_unique_orders
require_unique_orders(df)
if not df.take(1):
    raise ValueError("CSV source is empty")

# 3. Thống kê
print(f"Tổng số bản ghi (rows): {df.count()}")
print("=== THỐNG KÊ GIÁ TRỊ NULL THỜI GIAN/KHU VỰC ===")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# 4. Ghi trực tiếp ra S3 path (Bỏ qua saveAsTable)
print("Đang tiến hành ghi dữ liệu vào Delta Lake...")
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("s3a://lakehouse/warehouse/delivery_bronze")

print("Đã lưu bảng 'delivery_bronze' thành công!")